# Digitale Immobilienfinanzierung – Python-Prototypen

Dieses Notebook unterstützt das entwickelte End-to-End-Konzept für eine digitale private Immobilienfinanzierung.

Es enthält drei Anwendungen:

1. einen Annuitätenrechner zur Veranschaulichung einer Finanzierungsstruktur,
2. einen regelbasierten Routing-Prototyp zur Zuordnung von Finanzierungsfällen,
3. ein Prozesskostenmodell zur Abschätzung der wirtschaftlichen Wirkung des Fast-Track.

Die verwendeten Finanzierungsparameter sind illustrative eigene Annahmen und keine Marktprognose.
Der Routing-Prototyp dient ausschließlich der Prozessveranschaulichung und stellt kein produktives Kreditentscheidungsmodell dar.
Das Prozesskostenmodell ist eine Modellrechnung mit eigenen Annahmen und keine Kostenaussage für eine konkrete Bank.

## 1. Annuitätenrechner

Für ein illustratives Finanzierungsbeispiel werden folgende Annahmen verwendet:

- Kaufpreis: 500.000 €
- Eigenkapital: 100.000 €
- Darlehensbetrag: 400.000 €
- Sollzins: 3,50 % p. a.
- anfängliche Tilgung: 2,00 % p. a.
- Zinsbindung: 10 Jahre

Berechnet werden die monatliche Annuität sowie der Zins-, Tilgungs- und Restschuldverlauf über 120 Monate.

In [ ]:
kaufpreis = 500_000
eigenkapital = 100_000
darlehen = kaufpreis - eigenkapital

sollzins = 0.035
anfaengliche_tilgung = 0.02
zinsbindung_jahre = 10

monate = zinsbindung_jahre * 12
monatszins = sollzins / 12

monatsrate = darlehen * (sollzins + anfaengliche_tilgung) / 12

print(f"Darlehensbetrag: {darlehen:,.2f} €")
print(f"Monatliche Rate: {monatsrate:,.2f} €")

In [ ]:
restschuld = darlehen
tilgungsplan = []

for monat in range(1, monate + 1):
    zinsanteil = restschuld * monatszins
    tilgungsanteil = monatsrate - zinsanteil
    restschuld -= tilgungsanteil

    tilgungsplan.append({
        "Monat": monat,
        "Rate": monatsrate,
        "Zinsanteil": zinsanteil,
        "Tilgungsanteil": tilgungsanteil,
        "Restschuld": restschuld
    })

In [ ]:
import pandas as pd

tilgungsplan_df = pd.DataFrame(tilgungsplan)

tilgungsplan_df.head()

In [ ]:
restschuld_10_jahre = tilgungsplan_df.iloc[-1]["Restschuld"]
getilgter_betrag = darlehen - restschuld_10_jahre
gezahlte_zinsen = (
    tilgungsplan_df["Zinsanteil"].sum()
)

print(f"Monatliche Rate: {monatsrate:,.2f} €")
print(f"Getilgter Betrag nach 10 Jahren: {getilgter_betrag:,.2f} €")
print(f"Restschuld nach 10 Jahren: {restschuld_10_jahre:,.2f} €")
print(f"Gezahlte Zinsen in 10 Jahren: {gezahlte_zinsen:,.2f} €")

## 2. Regelbasierter Routing-Prototyp

Der Prototyp zeigt, wie Finanzierungsfälle anhand weniger qualitativer Kriterien unterschiedlichen Bearbeitungswegen zugeordnet werden können.

Verwendete Kriterien:

- Vollständigkeit der Daten
- Konsistenz der Daten
- standardisierte Einkommensstruktur
- ausreichend eindeutige Immobilienbewertung
- wesentliches Risikomerkmal

Die Regeln dienen ausschließlich der Prozesssteuerung. Sie ersetzen keine Kreditwürdigkeitsprüfung und kein validiertes Kreditentscheidungsmodell.

Negative Kreditentscheidungen werden im entwickelten Zielmodell nicht ausschließlich automatisiert finalisiert.

In [ ]:
def route_case(
    daten_vollstaendig,
    daten_konsistent,
    einkommen_standard,
    bewertung_sicher,
    wesentliches_risiko
):
    if wesentliches_risiko:
        return "Risikofall"

    if not all([
        daten_vollstaendig,
        daten_konsistent,
        einkommen_standard,
        bewertung_sicher
    ]):
        return "Prüffall"

    return "Fast-Track"

In [ ]:
testfaelle = [
    {
        "Fall": "A",
        "Daten vollständig": True,
        "Daten konsistent": True,
        "Einkommen standardisiert": True,
        "Bewertung sicher": True,
        "Wesentliches Risiko": False
    },
    {
        "Fall": "B",
        "Daten vollständig": True,
        "Daten konsistent": True,
        "Einkommen standardisiert": False,
        "Bewertung sicher": True,
        "Wesentliches Risiko": False
    },
    {
        "Fall": "C",
        "Daten vollständig": True,
        "Daten konsistent": True,
        "Einkommen standardisiert": True,
        "Bewertung sicher": True,
        "Wesentliches Risiko": True
    }
]

In [ ]:
for fall in testfaelle:
    fall["Routing"] = route_case(
        fall["Daten vollständig"],
        fall["Daten konsistent"],
        fall["Einkommen standardisiert"],
        fall["Bewertung sicher"],
        fall["Wesentliches Risiko"]
    )

routing_df = pd.DataFrame(testfaelle)

routing_df

## 3. Prozesskostenmodell

Das Modell schätzt ab, welche wirtschaftliche Wirkung ein höherer Fast-Track-Anteil auf die Prozesskosten je Antrag hat.

Verwendete Annahmen (eigene Setzung, keine Bankdaten):

- Antragsvolumen: 10.000 Anträge p. a.
- Vollkostensatz Bearbeitung: 75 €/h
- manuelle Bearbeitungszeit heute: 4–6 h je Antrag
- Bearbeitungszeit im Fast-Track: 1,0–1,5 h je Antrag
- Fast-Track-Anteil: 15 % (konservativ) bzw. 20 % (optimistisch)

Bewusst konservativ: Entlastet werden ausschließlich die Fast-Track-Fälle.
Für Prüffälle und Risikofälle wird die heutige Bearbeitungszeit unverändert angesetzt.

Die qualitative Wirkungsrichtung – Reduzierung von Durchlaufzeiten und Senkung von Abwicklungskosten durch
standardisierte Risikoprüfungen – folgt Bocken und Hagedorn (2018, S. 59). Die Zahlenwerte sind eigene Annahmen.

In [ ]:
antragsvolumen = 10_000
vollkostensatz = 75  # € je Stunde

szenarien = [
    {
        "Szenario": "konservativ",
        "Fast-Track-Anteil": 0.15,
        "Zeit heute (h)": 4.0,
        "Zeit Fast-Track (h)": 1.5
    },
    {
        "Szenario": "optimistisch",
        "Fast-Track-Anteil": 0.20,
        "Zeit heute (h)": 6.0,
        "Zeit Fast-Track (h)": 1.0
    }
]

In [ ]:
def prozesskosten(anteil_fast_track, zeit_heute, zeit_fast_track):
    kosten_heute = zeit_heute * vollkostensatz

    zeit_neu = (
        (1 - anteil_fast_track) * zeit_heute
        + anteil_fast_track * zeit_fast_track
    )
    kosten_neu = zeit_neu * vollkostensatz

    veraenderung = (kosten_neu - kosten_heute) / kosten_heute

    einsparung_pa = (
        antragsvolumen
        * anteil_fast_track
        * (zeit_heute - zeit_fast_track)
        * vollkostensatz
    )

    return kosten_heute, kosten_neu, veraenderung, einsparung_pa

In [ ]:
ergebnisse = []

for s in szenarien:
    kosten_heute, kosten_neu, veraenderung, einsparung_pa = prozesskosten(
        s["Fast-Track-Anteil"],
        s["Zeit heute (h)"],
        s["Zeit Fast-Track (h)"]
    )

    ergebnisse.append({
        "Szenario": s["Szenario"],
        "Fast-Track-Anteil": s["Fast-Track-Anteil"],
        "Kosten heute (€)": round(kosten_heute, 2),
        "Kosten neu (€)": round(kosten_neu, 2),
        "Veränderung": round(veraenderung * 100, 1),
        "Einsparung p. a. (€)": round(einsparung_pa, 0)
    })

kosten_df = pd.DataFrame(ergebnisse)

kosten_df

In [ ]:
for e in ergebnisse:
    print(
        f"{e['Szenario']:>13}: "
        f"{e['Kosten heute (€)']:>7,.2f} € -> {e['Kosten neu (€)']:>7,.2f} € je Antrag "
        f"({e['Veränderung']:>5.1f} %) | "
        f"Einsparung p. a.: {e['Einsparung p. a. (€)']:>10,.0f} €"
    )

In [ ]:
# Hebelwirkung: Effekt eines zusätzlichen Prozentpunkts Fast-Track-Anteil

for s in szenarien:
    delta_zeit = s["Zeit heute (h)"] - s["Zeit Fast-Track (h)"]
    effekt = antragsvolumen * 0.01 * delta_zeit * vollkostensatz

    print(
        f"{s['Szenario']:>13}: "
        f"je Prozentpunkt Fast-Track {effekt:,.0f} € p. a."
    )

## Interpretation

Der Annuitätenrechner zeigt, wie eine digitale Finanzierungsstrecke aus wenigen Eingabeparametern eine nachvollziehbare Finanzierungsstruktur ableiten kann.

Der Routing-Prototyp zeigt anschließend, wie standardisierte und abweichende Fälle unterschiedlichen Bearbeitungswegen zugeordnet werden können.

Das Prozesskostenmodell quantifiziert die wirtschaftliche Wirkung dieser Steuerung: Die Prozesskosten je Antrag sinken unter den
getroffenen Annahmen um rund 9 bis 17 Prozent. Entscheidend ist dabei nicht der Einzelwert, sondern der Hebel – jeder zusätzliche
Prozentpunkt Fast-Track-Anteil entlastet um rund 19.000 bis 38.000 € pro Jahr. Der tatsächliche Wert ist im Pilot gegen eine
erhobene Baseline zu messen.

Die Anwendung verdeutlicht damit das zentrale Prinzip des entwickelten Konzepts:

**Standardfälle digital beschleunigen – Abweichungen gezielt an qualifizierte Fachkräfte übergeben.**